In [ ]:
import os

In [ ]:
%pwd

In [ ]:
os.chdir('../')

In [ ]:
%pwd


In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list


In [ ]:
from KidneyDiseaseClassification.constants import *
from KidneyDiseaseClassification.utils.common import read_yaml,create_directories
import tensorflow as tf

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_training_config(self)->TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir,"kidney-ct-scan-image")
        create_directories(
            [training.root_dir]
        )

        training_config = TrainingConfig(
            root_dir =training.root_dir,
            trained_model_path = training.trained_model_path,
            updated_base_model_path=prepare_base_model.update_base_model_path,
            training_data=training_data,
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config


In [ ]:
import tensorflow as tf
import numpy as np
import math
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from KidneyDiseaseClassification.entity.config_entity import TrainingConfig

In [ ]:
class Training:
    def __init__(self,config:TrainingConfig):
        self.config = config
    
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split = 0.20
        )

        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[:-1],
            batch_size = self.config.params_batch_size,
            interpolation = 'bilinear'
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory = self.config.training_data,
            subset = "validation",
            shuffle = False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range = 15,
                horizontal_flip = True,
                width_shift_range = 0.1,
                height_shift_range = 0.1,
                shear_range = 0.1,
                zoom_range = 0.1,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory = self.config.training_data,
            subset = "training",
            shuffle = True,
            **dataflow_kwargs
        )
    
    def get_class_weights(self):
        return {
            0: 1.0,   # Normal — base weight
            1: 2.5    # Tumor — 2.5x penalty
        }

    def get_callbacks(self):
        return [
            tf.keras.callbacks.EarlyStopping(
                monitor = 'val_loss',
                patience = 5,
                restore_best_weights = True
            ),
            tf.keras.callbacks.ModelCheckpoint(
                filepath = str(self.config.trained_model_path),
                monitor = 'val_accuracy',
                save_best_only = True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor = 'val_loss',
                factor = 0.5,
                patience = 3,
                min_lr = 1e-7
            )
        ]
    
    def plot_training(self, history):
        fig, ax = plt.subplots(figsize=(10, 5))

        ax.plot(history.history['accuracy'], label='Train Accuracy')
        ax.plot(history.history['val_accuracy'], label='Val Accuracy')
        ax.plot(history.history['loss'], label='Train Loss')
        ax.plot(history.history['val_loss'], label='Val Loss')

        ax.set_title('Training History')
        ax.set_xlabel('Epoch')
        ax.legend()

        plt.tight_layout()
        plt.savefig('training_plot.png')
        print("Plot saved as training_plot.png")

    def train(self):
        self.step_per_epochs = math.ceil(self.train_generator.samples / self.train_generator.batch_size)
        self.validation_step = math.ceil(self.valid_generator.samples / self.valid_generator.batch_size)

        history = self.model.fit(
            self.train_generator,
            epochs = self.config.params_epochs,
            steps_per_epoch = self.step_per_epochs,
            validation_steps = self.validation_step,
            validation_data = self.valid_generator,
            class_weight = self.get_class_weights(), 
            callbacks = self.get_callbacks()  
        )

        self.plot_training(history)

        self.save_model(
            path=self.config.trained_model_path,
            model= self.model
        )

    @staticmethod
    def save_model(path:Path , model:tf.keras.Model):
        model.save(path)



In [ ]:
from KidneyDiseaseClassification.utils.exception import  CustomException
import sys

In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

In [ ]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
except Exception as e:
    raise CustomException(e,sys)